In [1]:
%pip install pyspark==4.0.1 findspark
%pip install numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import sys

os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"

In [3]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import split, col

spark = (
    SparkSession.builder
    .appName("big-data-programming-3")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.driver.memory", "8g")
    .config("spark.executor.memory", "8g")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

In [4]:
df = spark.read.csv(
    "./input/weatherData.csv",
    header=True,
    inferSchema=True,
)

df.show()

+-----------+---------+-----------------------+-----------------------+-----------------------+------------------------+-----------------------------+-----------------------------+------------------------------+---------------------+---------------------+----------------------+-------------+-----------------------+-------------------------+-------------------------+-------------------------------+-------------------------------+-------------------------------+-------------------+-------------------+
|location_id|     date|weather_code (wmo code)|temperature_2m_max (°C)|temperature_2m_min (°C)|temperature_2m_mean (°C)|apparent_temperature_max (°C)|apparent_temperature_min (°C)|apparent_temperature_mean (°C)|daylight_duration (s)|sunshine_duration (s)|precipitation_sum (mm)|rain_sum (mm)|precipitation_hours (h)|wind_speed_10m_max (km/h)|wind_gusts_10m_max (km/h)|wind_direction_10m_dominant (°)|shortwave_radiation_sum (MJ/m²)|et0_fao_evapotranspiration (mm)|            sunrise|           

In [5]:
print(df.count())

142371


In [6]:
df = df.withColumn("year", split(col("date"), "/").getItem(2))
df = df.withColumn("month", split(col("date"), "/").getItem(0))
df = df.withColumn("day", split(col("date"), "/").getItem(1))
df = df.select("year", "month", "day", "sunshine_duration (s)", "precipitation_hours (h)", "wind_speed_10m_max (km/h)",
               "et0_fao_evapotranspiration (mm)")

df = df.withColumnRenamed("sunshine_duration (s)", "sunshine_duration")
df = df.withColumnRenamed("precipitation_hours (h)", "precipitation_hours")
df = df.withColumnRenamed("wind_speed_10m_max (km/h)", "wind_speed")
df = df.withColumnRenamed("et0_fao_evapotranspiration (mm)", "evapotranspiration")

df = df[df["month"] == "5"]
df.show()

+----+-----+---+-----------------+-------------------+----------+------------------+
|year|month|day|sunshine_duration|precipitation_hours|wind_speed|evapotranspiration|
+----+-----+---+-----------------+-------------------+----------+------------------+
|2010|    5|  1|         40574.88|                 14|      10.3|              4.32|
|2010|    5|  2|          39600.0|                 10|       9.2|              4.29|
|2010|    5|  3|         39191.48|                  9|      11.9|              3.48|
|2010|    5|  4|         40046.57|                  9|      13.4|              3.76|
|2010|    5|  5|         40047.19|                  9|      13.8|              3.41|
|2010|    5|  6|         34909.82|                  8|      11.3|              3.58|
|2010|    5|  7|         38100.71|                 10|      12.4|              3.81|
|2010|    5|  8|         28622.78|                 17|      11.0|              3.27|
|2010|    5|  9|          5531.36|                 18|      14.8|

In [7]:
print(df.count())
1

12555


In [9]:
df.printSchema()

root
 |-- year: string (nullable = true)
 |-- month: string (nullable = true)
 |-- day: string (nullable = true)
 |-- sunshine_duration: double (nullable = true)
 |-- precipitation_hours: integer (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- evapotranspiration: double (nullable = true)



In [10]:
df = df.withColumn("year", col("year").cast("int"))
df = df.withColumn("month", col("month").cast("int"))
df = df.withColumn("day", col("day").cast("int"))
df.printSchema()

root
 |-- year: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- day: integer (nullable = true)
 |-- sunshine_duration: double (nullable = true)
 |-- precipitation_hours: integer (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- evapotranspiration: double (nullable = true)



In [11]:
df = df.drop("month")

df = df.withColumn("years_since_2010", col("year") - 2010)
df = df.drop("year")

df.show()

+---+-----------------+-------------------+----------+------------------+----------------+
|day|sunshine_duration|precipitation_hours|wind_speed|evapotranspiration|years_since_2010|
+---+-----------------+-------------------+----------+------------------+----------------+
|  1|         40574.88|                 14|      10.3|              4.32|               0|
|  2|          39600.0|                 10|       9.2|              4.29|               0|
|  3|         39191.48|                  9|      11.9|              3.48|               0|
|  4|         40046.57|                  9|      13.4|              3.76|               0|
|  5|         40047.19|                  9|      13.8|              3.41|               0|
|  6|         34909.82|                  8|      11.3|              3.58|               0|
|  7|         38100.71|                 10|      12.4|              3.81|               0|
|  8|         28622.78|                 17|      11.0|              3.27|               0|

In [12]:
# split 80/20
train_df, test_df = df.randomSplit([0.8, 0.2], seed=20242001)

In [13]:
feature_cols = ["sunshine_duration", "precipitation_hours", "wind_speed", "years_since_2010", "day"]
label_col = "evapotranspiration"

In [14]:
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline

assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

In [15]:
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

lr = LinearRegression(featuresCol="features", labelCol=label_col)
pipeline = Pipeline(stages=[assembler, scaler, lr])

paramGrid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .addGrid(lr.maxIter, [10, 50])
    .addGrid(lr.fitIntercept, [True, False])
    .addGrid(scaler.withMean, [True, False])
    .build()
)

evaluator = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3, parallelism=2)

linear_regression_model = cv.fit(train_df)

In [16]:
predictions = linear_regression_model.transform(test_df)
evaluator_rmse = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")

print("Test RMSE:", evaluator_rmse.evaluate(predictions))
print("Test R2:", evaluator_r2.evaluate(predictions))

Test RMSE: 0.5511523703508406
Test R2: 0.7958457915260858


In [17]:
model_evaluation = {
    "Linear Regression": {
        "RMSE": evaluator_rmse.evaluate(predictions),
        "R2": evaluator_r2.evaluate(predictions),
    }
}

In [18]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(featuresCol="raw_features", labelCol=label_col, seed=20242001)
pipeline = Pipeline(stages=[assembler, dt])

paramGrid = (
    ParamGridBuilder()
    .addGrid(dt.maxDepth, [5, 10, 20])
    .addGrid(dt.maxBins, [32, 64])
    .addGrid(dt.minInstancesPerNode, [1, 5])
    .build()
)

cv = CrossValidator(estimator=pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3, parallelism=2)

decision_tree_model = cv.fit(train_df)

In [19]:
predictions = decision_tree_model.transform(test_df)
evaluator_rmse = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")

print("Test RMSE:", evaluator_rmse.evaluate(predictions))
print("Test R2:", evaluator_r2.evaluate(predictions))

Test RMSE: 0.4722545988854257
Test R2: 0.8501118257526323


In [20]:
model_evaluation["Decision Tree"] = {
    "RMSE": evaluator_rmse.evaluate(predictions),
    "R2": evaluator_r2.evaluate(predictions),
}

In [21]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="raw_features", labelCol=label_col, seed=20242001)
pipeline = Pipeline(stages=[assembler, rf])

paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50])
    .addGrid(rf.maxDepth, [5, 10, 20])
    .addGrid(rf.maxBins, [32, 64])
    .addGrid(rf.featureSubsetStrategy, ["auto", "sqrt", "log2"])
    .build()
)

cv = CrossValidator(estimator=pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3, parallelism=2)
random_forest_model = cv.fit(train_df)

In [22]:
predictions = random_forest_model.transform(test_df)
evaluator_rmse = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")

print("Test RMSE:", evaluator_rmse.evaluate(predictions))
print("Test R2:", evaluator_r2.evaluate(predictions))

Test RMSE: 0.4164782800276724
Test R2: 0.8834265385338572


In [23]:
model_evaluation["Random Forest"] = {
    "RMSE": evaluator_rmse.evaluate(predictions),
    "R2": evaluator_r2.evaluate(predictions),
}

In [24]:
from pyspark.ml.regression import GBTRegressor

gbt_regressor = GBTRegressor(featuresCol="raw_features", labelCol=label_col, seed=20242001)

pipeline = Pipeline(stages=[assembler, gbt_regressor])

paramGrid = (
    ParamGridBuilder()
    .addGrid(gbt_regressor.maxIter, [20, 50])
    .addGrid(gbt_regressor.maxDepth, [3, 5])
    .addGrid(gbt_regressor.maxBins, [32])
    .addGrid(gbt_regressor.stepSize, [0.1, 0.2])
    .build()
)

cv = CrossValidator(estimator=pipeline, estimatorParamMaps=paramGrid, evaluator=evaluator, numFolds=3, parallelism=2)
gbt_model = cv.fit(train_df)

In [25]:
predictions = gbt_model.transform(test_df)
evaluator_rmse = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol=label_col, predictionCol="prediction", metricName="r2")

print("Test RMSE:", evaluator_rmse.evaluate(predictions))
print("Test R2:", evaluator_r2.evaluate(predictions))

Test RMSE: 0.428154894334019
Test R2: 0.8767982707134954


In [26]:
model_evaluation["Gradient Boosted Trees"] = {
    "RMSE": evaluator_rmse.evaluate(predictions),
    "R2": evaluator_r2.evaluate(predictions),
}

In [27]:
from tabulate import tabulate

print(tabulate(list(model_evaluation.items()), headers=["Model", "Evaluation Metrics"], tablefmt="github"))

| Model                  | Evaluation Metrics                                     |
|------------------------|--------------------------------------------------------|
| Linear Regression      | {'RMSE': 0.5511523703508406, 'R2': 0.7958457915260858} |
| Decision Tree          | {'RMSE': 0.4722545988854257, 'R2': 0.8501118257526323} |
| Random Forest          | {'RMSE': 0.4164782800276724, 'R2': 0.8834265385338572} |
| Gradient Boosted Trees | {'RMSE': 0.428154894334019, 'R2': 0.8767982707134955}  |


In [28]:
import os
import pandas as pd
import numpy as np
from pyspark.sql import functions as F

stats = df.select(
    F.min("sunshine_duration").alias("sun_min"),
    F.max("sunshine_duration").alias("sun_max"),
    F.min("precipitation_hours").alias("precip_min"),
    F.max("precipitation_hours").alias("precip_max"),
    F.min("wind_speed").alias("wind_min"),
    F.max("wind_speed").alias("wind_max"),
).collect()[0]

print(stats.wind_max)
print(stats.wind_min)

sunshine_duration_values = np.linspace(stats.sun_min, stats.sun_max, num=50)
precipitation_hours_values = np.linspace(stats.precip_min, stats.precip_max, num=50)
wind_speed_values = np.linspace(stats.wind_min, stats.wind_max, num=50)
day_values = np.arange(1, 32)
years_since_2010_values = [16]

print(wind_speed_values)

index = pd.MultiIndex.from_product(
    [day_values, sunshine_duration_values, precipitation_hours_values, wind_speed_values, years_since_2010_values],
    names=['day', 'sunshine_duration', 'precipitation_hours', 'wind_speed', 'years_since_2010']
)

simulated_data = pd.DataFrame(index=index).reset_index()
# save to csv
os.makedirs("./.tmp", exist_ok=True)
simulated_data.to_csv("./.tmp/simulated_weather_data.csv", index=False)

48.3
2.5
[ 2.5         3.43469388  4.36938776  5.30408163  6.23877551  7.17346939
  8.10816327  9.04285714  9.97755102 10.9122449  11.84693878 12.78163265
 13.71632653 14.65102041 15.58571429 16.52040816 17.45510204 18.38979592
 19.3244898  20.25918367 21.19387755 22.12857143 23.06326531 23.99795918
 24.93265306 25.86734694 26.80204082 27.73673469 28.67142857 29.60612245
 30.54081633 31.4755102  32.41020408 33.34489796 34.27959184 35.21428571
 36.14897959 37.08367347 38.01836735 38.95306122 39.8877551  40.82244898
 41.75714286 42.69183673 43.62653061 44.56122449 45.49591837 46.43061224
 47.36530612 48.3       ]


In [29]:
simulated_df = spark.read.csv(
    "./.tmp/simulated_weather_data.csv",
    header=True,
    inferSchema=True,
)

simulated_df.show()

+---+-----------------+-------------------+------------------+----------------+
|day|sunshine_duration|precipitation_hours|        wind_speed|years_since_2010|
+---+-----------------+-------------------+------------------+----------------+
|  1|              0.0|                0.0|               2.5|              16|
|  1|              0.0|                0.0|3.4346938775510205|              16|
|  1|              0.0|                0.0| 4.369387755102041|              16|
|  1|              0.0|                0.0| 5.304081632653061|              16|
|  1|              0.0|                0.0| 6.238775510204082|              16|
|  1|              0.0|                0.0| 7.173469387755102|              16|
|  1|              0.0|                0.0| 8.108163265306121|              16|
|  1|              0.0|                0.0| 9.042857142857143|              16|
|  1|              0.0|                0.0| 9.977551020408164|              16|
|  1|              0.0|                0

In [30]:
predictions = random_forest_model.transform(simulated_df)
predictions.show()

+---+-----------------+-------------------+------------------+----------------+--------------------+------------------+
|day|sunshine_duration|precipitation_hours|        wind_speed|years_since_2010|        raw_features|        prediction|
+---+-----------------+-------------------+------------------+----------------+--------------------+------------------+
|  1|              0.0|                0.0|               2.5|              16|[0.0,0.0,2.5,16.0...|2.7666753113553124|
|  1|              0.0|                0.0|3.4346938775510205|              16|[0.0,0.0,3.434693...|2.7666753113553124|
|  1|              0.0|                0.0| 4.369387755102041|              16|[0.0,0.0,4.369387...|2.7666753113553124|
|  1|              0.0|                0.0| 5.304081632653061|              16|[0.0,0.0,5.304081...|2.7666753113553124|
|  1|              0.0|                0.0| 6.238775510204082|              16|[0.0,0.0,6.238775...|2.7666753113553124|
|  1|              0.0|                0

In [31]:
low_evpotranspiration_predictions = predictions.filter(col("prediction") < 1.5)
low_evpotranspiration_predictions.show()

+---+-----------------+-------------------+------------------+----------------+--------------------+------------------+
|day|sunshine_duration|precipitation_hours|        wind_speed|years_since_2010|        raw_features|        prediction|
+---+-----------------+-------------------+------------------+----------------+--------------------+------------------+
| 18|              0.0|  20.57142857142857|10.912244897959184|              16|[0.0,20.571428571...|1.4968755772005773|
| 18|              0.0|  21.06122448979592|10.912244897959184|              16|[0.0,21.061224489...|1.4968755772005773|
| 18|              0.0| 21.551020408163264|12.781632653061225|              16|[0.0,21.551020408...| 1.487347243867244|
| 18|              0.0| 21.551020408163264|13.716326530612244|              16|[0.0,21.551020408...|1.4851758152958154|
| 18|              0.0|  22.04081632653061|12.781632653061225|              16|[0.0,22.040816326...| 1.487347243867244|
| 18|              0.0|  22.040816326530

In [32]:
print(low_evpotranspiration_predictions.count())

3294


In [33]:
low_evpotranspiration_predictions.select(
    "day", "sunshine_duration", "precipitation_hours", "wind_speed", "years_since_2010", "prediction"
).toPandas().to_csv("./output/low_evapotranspiration_predictions.csv", index=False)

In [36]:
mean_sunshine_duration = df.agg(F.mean("sunshine_duration")).collect()[0][0]
mean_precipitation_hours = df.agg(F.mean("precipitation_hours")).collect()[0][0]
mean_wind_speed = df.agg(F.mean("wind_speed")).collect()[0][0]

print("Mean Sunshine Duration:", round(mean_sunshine_duration, 2), "seconds")
print("Mean Precipitation:", round(mean_precipitation_hours, 2), "hours")
print("Mean Wind Speed:", round(mean_wind_speed, 2), "km/h")

Mean Sunshine Duration: 34302.53 seconds
Mean Precipitation: 8.01 hours
Mean Wind Speed: 17.85 km/h
